# A3.2 — Zero-shot Test for StyleTTS2-lite-vi (Kaggle Version)

**Mục đích**: Verify pretrained model `dangtr0408/StyleTTS2-lite-vi` có chấp nhận phoneme từ `viphoneme` (cách của bạn) không, TRƯỚC KHI tốn 20-30 giờ Kaggle GPU để fine-tune.

**Output kỳ vọng**: 5 samples × 2 phoneme methods × 2 audio outputs (sinh ra + reference Ngạn) = nhiều cặp audio để so sánh:
- Nếu **viphoneme audio nghe được tiếng Việt** (kể cả méo) → fine-tune sẽ work.
- Nếu **viphoneme audio nói lắp/sai âm vị nghiêm trọng** trong khi **espeak audio nghe ổn** → pretrained train bằng espeak → ta phải đổi pipeline sang espeak.
- Nếu **CẢ HAI đều méo** → có vấn đề khác (model load sai, sample rate sai, ...).

---

## Prerequisites — TRƯỚC KHI CHẠY NOTEBOOK NÀY

### Bước 0.1 — Tạo Kaggle Dataset chứa data test

Bạn cần upload lên Kaggle 1 dataset nhỏ gồm:
- **5 file audio Ngạn** (`.wav`, 24kHz mono) — chọn random 5 file từ `wavs/` của pipeline preprocess (tốt nhất là các file rõ ràng, không nhiễu).
- **1 file text** chứa text gốc tương ứng — file `filelist_train_clean.txt` (output của step0_clean_text) là đủ.

**Cách tạo dataset:**
1. Trên Kaggle: vào "Datasets" → "+ New Dataset"
2. Drag 5 file `.wav` + `filelist_train_clean.txt` vào upload area
3. Đặt tên dataset (vd: `ngan-zero-shot-test`)
4. Visibility: Private là OK
5. Click "Create"

### Bước 0.2 — Setup notebook

1. Tạo notebook mới trên Kaggle
2. Settings → **Accelerator**: chọn **GPU P100** (single GPU, đủ cho test, nhanh nhất). T4×2 cũng OK nhưng phí. KHÔNG dùng TPU.
3. Settings → **Internet**: BẬT (cần clone từ HuggingFace)
4. Sidebar "Input" → "+ Add Data" → tìm dataset `ngan-zero-shot-test` bạn vừa tạo → Add
5. Sau khi attach, dataset sẽ mount tại `/kaggle/input/ngan-zero-shot-test/`
6. Upload file `.ipynb` này lên (hoặc copy paste từng cell)
7. Chạy lần lượt từng cell từ trên xuống

---


## Bước 1 — Kiểm tra GPU và môi trường

In [ ]:
# Verify GPU
!nvidia-smi

import sys
import platform
print(f"\nPython : {sys.version}")
print(f"Platform: {platform.platform()}")

import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device  : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
# Verify PyTorch tương thích với GPU hiện tại
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Không có GPU! Hãy bật Accelerator trong Settings.")

capability = torch.cuda.get_device_capability(0)
gpu_name = torch.cuda.get_device_name(0)
sm_str = f"sm_{capability[0]}{capability[1]}"
supported_archs = torch.cuda.get_arch_list()

print(f"GPU                : {gpu_name}")
print(f"CUDA capability    : {sm_str}")
print(f"PyTorch hỗ trợ     : {supported_archs}")

if sm_str not in str(supported_archs):
    raise RuntimeError(
        f"\n❌ GPU {gpu_name} ({sm_str}) KHÔNG được PyTorch hiện tại hỗ trợ!\n"
        f"   PyTorch version: {torch.__version__}\n"
        f"   Supported architectures: {supported_archs}\n\n"
        f"GIẢI PHÁP: vào Settings (sidebar phải) → Accelerator → đổi sang 'GPU T4 x2'\n"
        f"           rồi click 'Save Session' để restart kernel với GPU mới."
    )
else:
    print(f"\n✅ GPU {sm_str} được PyTorch {torch.__version__} hỗ trợ. Tiếp tục.")

# Test 1 phép toán nhỏ trên GPU để verify thực sự work
try:
    x = torch.randn(10, 10, device='cuda')
    y = x @ x.T
    _ = y.sum().item()
    print("✅ GPU compute test PASS")
except Exception as e:
    raise RuntimeError(f"❌ GPU compute test FAIL: {e}")

## Bước 2 — Cài đặt dependencies

**Đã có sẵn trên Kaggle** (không cần cài lại): `torch`, `torchaudio`, `numpy`, `PyYAML`, `nltk`, `librosa`, `soundfile`, `matplotlib`.

**Cần cài thêm**:
- `munch` — utility để wrap dict thành object
- `noisereduce` — denoise audio reference (lite-vi yêu cầu)
- `phonemizer` + `espeakng-loader` — espeak backend (để so sánh với viphoneme)
- `viphoneme` + `vinorm` — phonemizer của user (giống step08 pipeline cũ)
- `gradio` — để app.py của lite-vi không bị lỗi import (KHÔNG dùng UI)

System: `espeak-ng` binary (cho phonemizer).

In [ ]:
# 1) System: cài espeak-ng
!apt-get install -y -qq espeak-ng espeak-ng-data 2>&1 | tail -3

# 2) Python deps cho lite-vi
!pip install -q munch noisereduce phonemizer espeakng-loader 2>&1 | tail -3

# 3) Python deps cho viphoneme
!pip install -q underthesea eng_to_ipa 2>&1 | tail -3
!pip install -q viphoneme vinorm 2>&1 | tail -3

# 4) Patch Python 3.12 cho vinorm/viphoneme
# Python 3.12 đã remove module imp, nhưng vinorm cũ vẫn import imp.
import sys, types, importlib.util

def install_imp_compat_shim():
    if "imp" in sys.modules:
        return

    imp_mod = types.ModuleType("imp")

    def find_module(name, path=None):
        spec = importlib.util.find_spec(name, path)
        if spec is None:
            raise ImportError(f"No module named {name!r}")

        if spec.submodule_search_locations:
            pathname = list(spec.submodule_search_locations)[0]
        else:
            pathname = spec.origin

        return None, pathname, ("", "", 5)

    imp_mod.find_module = find_module
    sys.modules["imp"] = imp_mod

install_imp_compat_shim()

# 5) Verify imports an toàn
import munch
import noisereduce
import phonemizer
import espeakng_loader

print("munch          :", getattr(munch, "__version__", "OK"))
print("noisereduce    :", getattr(noisereduce, "__version__", "OK"))
print("phonemizer     :", getattr(phonemizer, "__version__", "OK"))
print("espeakng_loader: OK")

# 6) Verify underthesea
try:
    import underthesea
    print("underthesea    :", getattr(underthesea, "__version__", "OK"))
except Exception as e:
    print("underthesea import FAILED:", repr(e))
    raise

# 7) Verify vinorm + viphoneme
import vinorm

def _mock_tts_norm(text, *args, **kwargs):
    return str(text).lower().strip()

vinorm.TTSnorm = _mock_tts_norm
vinorm.TTSrawUpper = lambda t, *a, **k: str(t).strip()

import viphoneme
viphoneme.TTSnorm = _mock_tts_norm

print("vinorm         : OK")
print("viphoneme      : OK")

# 8) Test thử viphoneme
from viphoneme import vi2IPA_split

test_text = "Xin chào, hôm nay trời đẹp quá."
print("test text      :", test_text)
print("viphoneme test :", vi2IPA_split(test_text, " "))

# 9) Verify espeak-ng binary
!espeak-ng --version

## Bước 3 — Clone StyleTTS2-lite-vi từ HuggingFace

Bao gồm source code (models.py, Modules/, meldataset.py) + model weights (~570MB) + config.yaml.

In [ ]:
import os

# Working directory cho session
WORK_DIR = "/kaggle/working"
REPO_DIR = f"{WORK_DIR}/StyleTTS2-lite-vi"

# Cài git-lfs (cần để pull file model.pth lớn)
!apt-get install -y -qq git-lfs 2>&1 | tail -1
!git lfs install

# Clone
os.chdir(WORK_DIR)
if not os.path.exists(REPO_DIR):
    !git clone https://huggingface.co/dangtr0408/StyleTTS2-lite-vi
else:
    print(f"Repo đã clone tại: {REPO_DIR}")

os.chdir(REPO_DIR)
!ls -lah Models/

# Đảm bảo model.pth đã pull đủ (nếu lfs chưa pull xong)
if not os.path.exists(f"{REPO_DIR}/Models/model.pth") or os.path.getsize(f"{REPO_DIR}/Models/model.pth") < 100_000_000:
    print("\nFile model.pth thiếu hoặc nhỏ — pulling lại bằng git lfs...")
    !cd {REPO_DIR} && git lfs pull
    !ls -lah Models/


## Bước 4 — Detect data test bạn đã upload

Notebook sẽ tự tìm:
- Các file `.wav` trong `/kaggle/input/`
- File text (`filelist_train_clean.txt` hoặc tên có chứa "filelist") trong `/kaggle/input/`

**Nếu output KHÔNG tìm thấy file nào**, kiểm tra lại đã attach dataset chưa (sidebar "+ Add Data").

In [ ]:
import glob

INPUT_BASE = "/kaggle/input"

# Tìm tất cả wav và text file
wav_files = sorted(glob.glob(f"{INPUT_BASE}/**/*.wav", recursive=True))
text_files = sorted(glob.glob(f"{INPUT_BASE}/**/filelist*.txt", recursive=True))
# Fallback: nếu không thấy filelist, lấy bất kỳ .txt nào
if not text_files:
    text_files = sorted(glob.glob(f"{INPUT_BASE}/**/*.txt", recursive=True))

print(f"Tìm thấy {len(wav_files)} file .wav:")
for w in wav_files[:10]:
    size_mb = os.path.getsize(w) / 1e6
    print(f"  {w}  ({size_mb:.2f} MB)")

print(f"\nTìm thấy {len(text_files)} file .txt:")
for t in text_files:
    print(f"  {t}")

if not wav_files:
    raise RuntimeError(
        "Không tìm thấy file wav nào trong /kaggle/input/!\n"
        "Bạn cần Attach Dataset chứa 5 file audio Ngạn vào notebook này."
    )
if not text_files:
    raise RuntimeError(
        "Không tìm thấy file text (filelist) trong /kaggle/input/!\n"
        "Hãy upload filelist_train_clean.txt vào cùng dataset."
    )

TEST_TEXT_FILE = text_files[0]
print(f"\nSẽ dùng text file: {TEST_TEXT_FILE}")


## Bước 5 — Load config & build symbol_dict (vocab 189)

Logic GIỐNG HỆT `inference.py` của lite-vi, đã verify ở file A2.

In [ ]:
import yaml

CONFIG_PATH = f"{REPO_DIR}/Models/config.yaml"
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

symbols = (
    list(config["symbol"]["pad"])
    + list(config["symbol"]["punctuation"])
    + list(config["symbol"]["letters"])
    + list(config["symbol"]["letters_ipa"])
    + list(config["symbol"]["extend"])
)
symbol_dict = {symbols[i]: i for i in range(len(symbols))}
n_token = len(symbol_dict) + 1

print(f"Loaded {len(symbol_dict)} unique symbols, n_token = {n_token}")
assert n_token == 189, f"Expected 189 tokens, got {n_token}"
print("OK — vocab matches lite-vi pretrained")


## Bước 6 — Build 4 components + Load checkpoint weights

Logic giống `inference.py.__init__` và `__load_models`. Build từng module riêng để dễ debug nếu lỗi.

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

import torch
from munch import Munch
from collections import OrderedDict
from models import ProsodyPredictor, TextEncoder, StyleEncoder
from Modules.hifigan import Decoder

def recursive_munch(d):
    if isinstance(d, dict):
        return Munch((k, recursive_munch(v)) for k, v in d.items())
    elif isinstance(d, list):
        return [recursive_munch(v) for v in d]
    return d

device = "cuda" if torch.cuda.is_available() else "cpu"
args = recursive_munch(config["model_params"])
args.n_token = n_token

assert args.decoder.type == "hifigan", f"Expected hifigan, got {args.decoder.type}"

# Build 4 components
decoder = Decoder(
    dim_in=args.hidden_dim,
    style_dim=args.style_dim,
    dim_out=args.n_mels,
    resblock_kernel_sizes=args.decoder.resblock_kernel_sizes,
    upsample_rates=args.decoder.upsample_rates,
    upsample_initial_channel=args.decoder.upsample_initial_channel,
    resblock_dilation_sizes=args.decoder.resblock_dilation_sizes,
    upsample_kernel_sizes=args.decoder.upsample_kernel_sizes,
).to(device)

predictor = ProsodyPredictor(
    style_dim=args.style_dim,
    d_hid=args.hidden_dim,
    nlayers=args.n_layer,
    max_dur=args.max_dur,
    dropout=args.dropout,
).to(device)

text_encoder = TextEncoder(
    channels=args.hidden_dim,
    kernel_size=5,
    depth=args.n_layer,
    n_symbols=args.n_token,
).to(device)

style_encoder = StyleEncoder(
    dim_in=args.dim_in,
    style_dim=args.style_dim,
    max_conv_dim=args.hidden_dim,
).to(device)

model = {
    "decoder": decoder,
    "predictor": predictor,
    "text_encoder": text_encoder,
    "style_encoder": style_encoder,
}

# Load weights
CHECKPOINT_PATH = f"{REPO_DIR}/Models/inference/model.pth"
print(f"Loading checkpoint từ {CHECKPOINT_PATH}...")
params_whole = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
params = params_whole["net"]
params = {k: v for k, v in params.items() if k in model.keys()}

total_params = 0
for key in model:
    try:
        model[key].load_state_dict(params[key])
    except Exception:
        # Fallback: strip 'module.' prefix (DataParallel)
        new_state_dict = OrderedDict()
        for k, v in params[key].items():
            name = k[7:] if k.startswith("module.") else k
            new_state_dict[name] = v
        model[key].load_state_dict(new_state_dict, strict=False)
    model[key].eval()
    n_params = sum(p.numel() for p in model[key].parameters())
    total_params += n_params
    print(f"  {key:14s}: {n_params/1e6:6.2f}M params")

print(f"  {'TOTAL':14s}: {total_params/1e6:6.2f}M params")
print("\nModel loaded successfully")

## Bước 7 — Helper functions

Copy nguyên logic từ `inference.py` của lite-vi: `compute_style()` (trích style từ audio reference) và `inference_from_phoneme()` (bypass espeak để nhận phoneme từ bất kỳ phonemizer nào).

In [ ]:
import librosa
import noisereduce as nr
import torchaudio
from nltk.tokenize import word_tokenize
import nltk

# Setup NLTK
try:
    nltk.data.find("tokenizers/punkt_tab")
except LookupError:
    nltk.download("punkt_tab", quiet=True)
try:
    nltk.data.find("tokenizers/punkt")
except LookupError:
    nltk.download("punkt", quiet=True)

# Mel transform (same as inference.py)
_TO_MEL = torchaudio.transforms.MelSpectrogram(
    n_mels=80, n_fft=2048, win_length=1200, hop_length=300
)

def wave_to_mel(wave_np):
    """numpy wave -> mel tensor, same as inference.py preprocess."""
    mean, std = -4, 4
    wave_tensor = torch.from_numpy(wave_np).float()
    mel = _TO_MEL(wave_tensor)
    mel = (torch.log(1e-5 + mel.unsqueeze(0)) - mean) / std
    return mel

@torch.no_grad()
def compute_style(audio_path, denoise=0.3, split_dur=2):
    """
    Compute style vector từ audio reference.
    - denoise: 0.0-1.0, % blend với noisereduce output
    - split_dur: chia audio thành các đoạn split_dur (giây), lấy style trung bình.
                Đặt 0 để dùng full audio (không chia).
    """
    max_samples = 24000 * 20  # max 20s
    wave, sr = librosa.load(audio_path, sr=24000)
    audio, _ = librosa.effects.trim(wave, top_db=30)
    if len(audio) > max_samples:
        audio = audio[:max_samples]
    if denoise > 0:
        audio_dn = nr.reduce_noise(y=audio, sr=24000, n_fft=2048, win_length=1200, hop_length=300)
        audio = audio * (1 - denoise) + audio_dn * denoise

    if split_dur > 0 and len(audio) / 24000 >= 4:
        jump = 24000 * split_dur
        total_len = len(audio)
        mel = wave_to_mel(audio[0:jump]).to(device)
        ref_s = style_encoder(mel.unsqueeze(1))
        count = 1
        for i in range(jump, total_len, jump):
            if i + jump >= total_len:
                left_dur = (total_len - i) / 24000
                if left_dur >= 1:
                    mel = wave_to_mel(audio[i:total_len]).to(device)
                    ref_s += style_encoder(mel.unsqueeze(1))
                    count += 1
                continue
            mel = wave_to_mel(audio[i:i+jump]).to(device)
            ref_s += style_encoder(mel.unsqueeze(1))
            count += 1
        ref_s /= count
    else:
        mel = wave_to_mel(audio).to(device)
        ref_s = style_encoder(mel.unsqueeze(1))
    return ref_s


def length_to_mask(lengths):
    mask = torch.arange(lengths.max()).unsqueeze(0).expand(lengths.shape[0], -1).type_as(lengths)
    mask = torch.gt(mask + 1, lengths.unsqueeze(1))
    return mask


def replace_outliers_zscore(tensor, threshold=3.0, factor=0.95):
    mean = tensor.mean()
    std = tensor.std()
    z = (tensor - mean) / std
    outlier_mask = torch.abs(z) > threshold
    sign = torch.sign(tensor - mean)
    replacement = mean + sign * (threshold * std * factor)
    result = tensor.clone()
    result[outlier_mask] = replacement[outlier_mask]
    return result


# TextCleaner (cần import sau khi sys.path đã có REPO_DIR)
from meldataset import TextCleaner
cleaner = TextCleaner(symbol_dict, debug=True)


@torch.no_grad()
def inference_from_phoneme(phoneme_str, ref_s, speed=1.0, t=0.1):
    """
    Inference từ chuỗi phoneme đã chuẩn bị sẵn (bypass espeak).
    Logic GIỐNG HỆT __inference của inference.py, chỉ thay phần phonemize.
    """
    speed = min(max(speed, 0.0001), 2)

    # word_tokenize + cleaner -> tokens (giống y nguyên inference.py)
    phn = " ".join(word_tokenize(phoneme_str))
    tokens = cleaner(phn)
    if len(tokens) == 0:
        raise ValueError(f"Phoneme rỗng sau khi cleaner — có thể tất cả ký tự không nằm trong vocab!\n  Phoneme: {phoneme_str!r}")
    tokens.insert(0, 0)
    tokens.append(0)
    tokens = torch.LongTensor(tokens).to(device).unsqueeze(0)

    input_lengths = torch.LongTensor([tokens.shape[-1]]).to(device)
    text_mask = length_to_mask(input_lengths).to(device)

    t_en = text_encoder(tokens, input_lengths, text_mask)
    s = ref_s.to(device)

    d = predictor.text_encoder(t_en, s, input_lengths, text_mask)
    x, _ = predictor.lstm(d)
    duration = predictor.duration_proj(x)
    duration = torch.sigmoid(duration).sum(axis=-1)

    # Stabilize duration (smooth)
    dur_stats = torch.empty(duration.shape).normal_(
        mean=duration.mean(), std=duration.std()
    ).to(device)
    duration = duration * (1 - t) + dur_stats * t
    duration[:, 1:-2] = replace_outliers_zscore(duration[:, 1:-2])
    duration /= speed

    pred_dur = torch.round(duration.squeeze()).clamp(min=1)
    pred_aln_trg = torch.zeros(input_lengths, int(pred_dur.sum().data))
    c_frame = 0
    for i in range(pred_aln_trg.size(0)):
        pred_aln_trg[i, c_frame:c_frame + int(pred_dur[i].data)] = 1
        c_frame += int(pred_dur[i].data)
    alignment = pred_aln_trg.unsqueeze(0).to(device)

    en = (d.transpose(-1, -2) @ alignment)
    F0_pred, N_pred = predictor.F0Ntrain(en, s)
    asr = (t_en @ pred_aln_trg.unsqueeze(0).to(device))

    out = decoder(asr, F0_pred, N_pred, s)
    wav = out.squeeze().cpu().numpy()
    wav = wav[4000:-4000]  # remove weird silent tokens at edges (theo inference.py)
    return wav


print("Helper functions ready.")


## Bước 8 — Setup 2 phonemizer (viphoneme + espeak-ng) để so sánh

- **Way 1 — viphoneme**: cách của user (giống `step08_phonemize.py` + replace `_` → space ở file A1). Dùng monkey-patch vinorm.
- **Way 2 — espeak-ng**: cách gốc lite-vi dùng trong `inference.py`. Đây là baseline.

In [ ]:
# ====== Way 1: viphoneme (giống user pipeline) ======

# Monkey-patch vinorm như step08 cũ — để consistent với data đã clean
import vinorm
def _mock_tts_norm(text, *args, **kwargs):
    return str(text).lower().strip()
vinorm.TTSnorm = _mock_tts_norm
vinorm.TTSrawUpper = lambda t, *a, **k: str(t).strip()

import viphoneme
viphoneme.TTSnorm = _mock_tts_norm

from viphoneme import vi2IPA_split
import re

def phonemize_viphoneme(text):
    """Cách của user: viphoneme + replace _ -> space."""
    phn = vi2IPA_split(text, " ")
    phn = phn.replace("_", " ")
    phn = re.sub(r"\s+", " ", phn).strip()
    return phn


# ====== Way 2: espeak-ng (cách gốc lite-vi) ======
import phonemizer
import sys as _sys
if _sys.platform.startswith("win"):
    try:
        from phonemizer.backend.espeak.wrapper import EspeakWrapper
        import espeakng_loader
        EspeakWrapper.set_library(espeakng_loader.get_library_path())
    except Exception as e:
        print(f"(warning) espeakng_loader setup: {e}")

def phonemize_espeak(text, lang="vi"):
    backend = phonemizer.backend.EspeakBackend(
        language=lang,
        preserve_punctuation=True,
        with_stress=True,
        language_switch="remove-flags",
    )
    return backend.phonemize([text])[0]


# ====== Test 2 cách với 1 câu mẫu ======
test_text = "Xin chào, hôm nay trời đẹp quá."
print(f"Text gốc        : {test_text}")
print(f"viphoneme output: {phonemize_viphoneme(test_text)!r}")
print(f"espeak    output: {phonemize_espeak(test_text)!r}")


## Bước 9 — Load text samples & match với audio reference

Đọc filelist clean → match audio name → tạo list 5 samples để test.

In [ ]:
# Load tất cả dòng từ filelist
text_records = []
with open(TEST_TEXT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split("|", 1)
        if len(parts) != 2:
            continue
        wav_path, text = parts[0].strip(), parts[1].strip()
        text_records.append({
            "wav_name": os.path.basename(wav_path.replace("\\", "/")),
            "text": text,
        })

print(f"Đã đọc {len(text_records)} dòng từ filelist.")

# Map wav_name -> full path
wav_dict = {os.path.basename(w): w for w in wav_files}
print(f"Có {len(wav_dict)} file audio đã upload.")

# Tìm samples có cả text và audio
selected = []
for rec in text_records:
    if rec["wav_name"] in wav_dict:
        selected.append({
            **rec,
            "wav_path": wav_dict[rec["wav_name"]],
        })
        if len(selected) >= 5:
            break

print(f"\nMatched {len(selected)} samples để test:")
for i, s in enumerate(selected, 1):
    print(f"  {i}. {s['wav_name']}")
    print(f"     text: {s['text'][:100]}{'...' if len(s['text']) > 100 else ''}")

if not selected:
    raise RuntimeError(
        "KHÔNG match được audio nào với filelist text!\n"
        "Hãy đảm bảo bạn upload audio có tên trùng với cột 1 của filelist.\n"
        "Ví dụ: filelist ghi 'wavs/ngan_00001.wav|...' thì cần upload ngan_00001.wav."
    )


## Bước 10 — MAIN LOOP: chạy inference cho 5 samples × 2 phonemizer

Output là 15 đoạn audio (5 viphoneme + 5 espeak + 5 reference Ngạn gốc) hiển thị trực tiếp dưới notebook để bạn nghe so sánh.

**Cách judge**:
- Nghe lần lượt 3 audio mỗi sample.
- Reference Ngạn gốc: là target — bạn KHÔNG kỳ vọng audio sinh ra giống y hệt vì model chưa fine-tune.
- viphoneme audio: kỳ vọng **nghe ra được tiếng Việt**, có thể giọng khác Ngạn.
- espeak audio: tương tự, để đối chiếu xem cách nào âm vị chuẩn hơn.

In [ ]:
import soundfile as sf
from IPython.display import Audio, display, Markdown, HTML

OUTPUT_DIR = "/kaggle/working/zero_shot_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

for idx, sample in enumerate(selected, 1):
    display(HTML(f"<hr><h3>Sample {idx}/{len(selected)}: <code>{sample['wav_name']}</code></h3>"))
    display(Markdown(f"**Text gốc**: {sample['text']}"))

    # Compute style từ audio Ngạn
    try:
        ref_s = compute_style(sample["wav_path"], denoise=0.3, split_dur=2)
    except Exception as e:
        display(Markdown(f"❌ Style compute FAILED: `{e}`"))
        continue

    # ---- Reference audio (Ngạn gốc) ----
    display(Markdown("**A. Audio reference Ngạn (gốc):**"))
    display(Audio(sample["wav_path"]))

    # ---- Way 1: viphoneme ----
    display(Markdown("---"))
    display(Markdown("**B. Inference dùng `viphoneme`**"))
    try:
        phn_vi = phonemize_viphoneme(sample["text"])
        display(Markdown(f"`Phoneme: {phn_vi[:200]}{'...' if len(phn_vi) > 200 else ''}`"))
        wav_vi = inference_from_phoneme(phn_vi, ref_s)
        # Normalize amplitude
        peak = max(1e-9, abs(wav_vi).max())
        wav_vi_norm = wav_vi / peak
        out_path_vi = f"{OUTPUT_DIR}/sample_{idx:02d}_viphoneme.wav"
        sf.write(out_path_vi, wav_vi_norm, 24000)
        display(Audio(out_path_vi))
    except Exception as e:
        display(Markdown(f"❌ viphoneme inference FAILED: `{type(e).__name__}: {e}`"))

    # ---- Way 2: espeak ----
    display(Markdown("---"))
    display(Markdown("**C. Inference dùng `espeak-ng`**"))
    try:
        phn_es = phonemize_espeak(sample["text"])
        display(Markdown(f"`Phoneme: {phn_es[:200]}{'...' if len(phn_es) > 200 else ''}`"))
        wav_es = inference_from_phoneme(phn_es, ref_s)
        peak = max(1e-9, abs(wav_es).max())
        wav_es_norm = wav_es / peak
        out_path_es = f"{OUTPUT_DIR}/sample_{idx:02d}_espeak.wav"
        sf.write(out_path_es, wav_es_norm, 24000)
        display(Audio(out_path_es))
    except Exception as e:
        display(Markdown(f"❌ espeak inference FAILED: `{type(e).__name__}: {e}`"))

print("\n=== TEST HOÀN TẤT ===")
print(f"Output files saved tại: {OUTPUT_DIR}")


## Bước 11 — Tải về & đánh giá

Outputs được lưu tại `/kaggle/working/zero_shot_outputs/`. File trong `/kaggle/working/` sẽ được Kaggle TỰ ĐỘNG zip lại khi bạn click **"Save Version"** → "Save & Run All" để có thể download.

Hoặc click vào file trong sidebar phải (panel "Output") để download từng file `.wav`.

In [ ]:
# List output files
!ls -lah /kaggle/working/zero_shot_outputs/

# Tổng kết để tiện báo lại cho Claude
print("\n" + "=" * 50)
print("BÁO CÁO ZERO-SHOT TEST")
print("=" * 50)
print(f"GPU sử dụng       : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Số samples test   : {len(selected)}")
print(f"Số audio output   : {len([f for f in os.listdir(OUTPUT_DIR) if f.endswith('.wav')])}")
print(f"Output directory  : {OUTPUT_DIR}")


## Bước 12 — Cách diễn giải kết quả

### Nghe 5 samples và đánh giá theo bảng sau:

| Kết quả | Kết luận | Hành động tiếp theo |
|---------|----------|---------------------|
| **viphoneme NGHE ĐƯỢC tiếng Việt** (kể cả méo, giọng khác Ngạn) | Pretrained chấp nhận format viphoneme → fine-tune sẽ work | ✅ Đi tiếp Kaggle fine-tune (file B2) |
| **viphoneme ra tạp âm/nói lắp**, espeak nghe rõ tiếng Việt | Pretrained TRAIN BẰNG espeak → format mismatch | ⚠️ Cần đổi pipeline: rephonemize toàn bộ data bằng espeak |
| **CẢ HAI đều méo nặng** | Có vấn đề khác (model load lỗi, hoặc sample rate sai) | 🛑 Quay lại debug — báo Claude |
| **viphoneme + espeak NGANG NHAU** | Cả hai format đều OK | ✅ Đi tiếp với viphoneme (vì user đã có pipeline cũ) |

### Cách báo cáo cho Claude

Sau khi nghe xong 5 samples, gửi tôi:
1. **Đánh giá tổng quan**: viphoneme có nghe được tiếng Việt không (yes/no/partial)?
2. **Đánh giá so sánh viphoneme vs espeak**: cái nào rõ hơn?
3. **Có lỗi gì xuất hiện trong cell output không?** Đặc biệt là warning "UNKNOWN IPA CHARACTERS" từ TextCleaner.
4. **(Optional)**: 1-2 audio file download về để tôi không nghe được nhưng bạn mô tả chất lượng (clear/distorted/silent/...)

Sau khi confirm, ta sẽ chuyển sang **file B2 — Kaggle fine-tune notebook**.